<a href="https://colab.research.google.com/github/skyestaq/spin_to_strava/blob/main/spin_to_strava_v_260301.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚴 Spin Class → Strava Uploader

This Google Colab notebook allows you to upload a photo of your spin bike screen and automatically create a Strava activity.

## Setup

**First time setup:**
1. Create `config.json` in Google Drive at `My Drive/spin_to_strava/config.json`.
2. Run Cell 1 to install dependencies.
3. Run Cell 2 to verify credentials load.

**Each use:**
1. Run all cells.
2. Upload your spin bike screen photo when prompted.

## Strava Token Refresh

If you encounter authentication errors or need to reauthorize Strava access, refer to the "Strava Token Refresh Procedure" section in the notebook (Cell `rQv3P10eV98B`).

**Version:** 26.03.01

In [ ]:
#@title Cell 1: Install Dependencies

!pip install openai requests -q
print("✅ Dependencies installed.")

✅ Dependencies installed.


In [ ]:
from google.colab import drive
import json
import os
os.environ['TZ'] = 'America/New_York'  # Set to your timezone

drive.mount('/content/drive')

CONFIG_FILE = '/content/drive/MyDrive/spin_to_strava/config.json'

if not os.path.exists(CONFIG_FILE):
    raise Exception(f"Config file not found at {CONFIG_FILE}. Create it with your credentials.")

with open(CONFIG_FILE, 'r') as f:
    config = json.load(f)

required = ['strava_client_id', 'strava_client_secret', 'strava_refresh_token', 'openai_api_key']
missing = [k for k in required if not config.get(k)]
if missing:
    raise Exception(f"Config file is missing: {', '.join(missing)}")

print("✅ Credentials loaded from Google Drive.")
print(f"   Strava Client ID: {config['strava_client_id']}")
print(f"   (other credentials hidden)")

print("\n⚠️ Remember to delete your 'Bike Indoor' activity from your watch before uploading your ride.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Credentials loaded from Google Drive.
   Strava Client ID: 188892
   (other credentials hidden)

⚠️ Remember to delete your 'Bike Indoor' activity from your watch before uploading your ride.


In [ ]:
#@title Cell 3: Upload Spin Class Photo

import openai
import requests
import json
import base64
import re
import os
from datetime import datetime, timedelta
from google.colab import files

# Ensure config is loaded fresh from Drive
CONFIG_FILE = '/content/drive/MyDrive/spin_to_strava/config.json'
with open(CONFIG_FILE, 'r') as f:
    config = json.load(f)

openai.api_key = config['openai_api_key']

# === Helper Functions ===

def get_strava_access_token():
    """Exchange refresh token for a fresh access token."""
    global config
    # Re-read config to ensure we have the manually updated refresh token
    with open(CONFIG_FILE, 'r') as f:
        config = json.load(f)

    response = requests.post(
        "https://www.strava.com/oauth/token",
        data={
            "client_id": config['strava_client_id'],
            "client_secret": config['strava_client_secret'],
            "refresh_token": config['strava_refresh_token'],
            "grant_type": "refresh_token"
        }
    )
    if response.status_code != 200:
        error_msg = response.text
        if "invalid" in error_msg.lower():
            raise Exception("Strava token refresh failed. Your refresh token is invalid. Get a new one and update config.json.")
        raise Exception(f"Strava token refresh failed: {error_msg}")

    data = response.json()

    # Save the new refresh token back to config file
    config['strava_refresh_token'] = data['refresh_token']
    with open(CONFIG_FILE, 'w') as f:
        json.dump(config, f, indent=2)

    return data["access_token"]

def extract_workout_data(image_bytes):
    """Use GPT-4o to extract workout data from spin bike screen image."""
    base64_image = base64.b64encode(image_bytes).decode('utf-8')

    try:
        response = openai.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": """Extract the workout data from this spin bike screen image.
Return ONLY a JSON object with these fields (use null if not visible):
{
  "time_seconds": <total time in seconds, e.g. 39:46 = 2386>,
  "distance_miles": <distance in miles as a number>,
  "calories": <total calories as a number>,
  "avg_watts": <average watts as a number>,
  "avg_rpm": <average RPM/cadence as a number>,
  "avg_speed_mph": <average speed in mph as a number>
}
Return only the JSON, no other text."""
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            max_tokens=300
        )
    except Exception as e:
        raise Exception(f"OpenAI API error: {str(e)}. Check your API key.")

    result_text = response.choices[0].message.content.strip()
    result_text = re.sub(r'^```json\s*', '', result_text)
    result_text = re.sub(r'\s*```$', '', result_text)

    try:
        return json.loads(result_text)
    except json.JSONDecodeError:
        raise Exception(f"Failed to parse OCR result. Raw response: {result_text}")

def parse_datetime(input_str):
    """Parse flexible date/time input."""
    input_str = input_str.strip().lower()
    now = datetime.now()

    def parse_time(time_str):
        match = re.match(r'^(\d{1,2})(?::(\d{2}))?\s*(am|pm)?$', time_str, re.IGNORECASE)
        if not match:
            return None
        hour = int(match.group(1))
        minute = int(match.group(2)) if match.group(2) else 0
        ampm = match.group(3)

        if ampm:
            if ampm.lower() == 'pm' and hour != 12:
                hour += 12
            elif ampm.lower() == 'am' and hour == 12:
                hour = 0

        return hour, minute

    if input_str.startswith('today'):
        time_part = input_str.replace('today', '').strip()
        parsed = parse_time(time_part)
        if parsed:
            return now.replace(hour=parsed[0], minute=parsed[1], second=0, microsecond=0)

    elif input_str.startswith('yesterday'):
        time_part = input_str.replace('yesterday', '').strip()
        parsed = parse_time(time_part)
        if parsed:
            yesterday = now - timedelta(days=1)
            return yesterday.replace(hour=parsed[0], minute=parsed[1], second=0, microsecond=0)

    try:
        return datetime.strptime(input_str, "%Y-%m-%d %H:%M")
    except ValueError:
        pass

    parsed = parse_time(input_str)
    if parsed:
        return now.replace(hour=parsed[0], minute=parsed[1], second=0, microsecond=0)

    return None

def create_strava_activity(access_token, name, start_time, elapsed_seconds, distance_meters, description):
    """Create a Strava activity."""
    payload = {
        "name": name,
        "type": "Ride",
        "sport_type": "VirtualRide",
        "start_date_local": start_time,
        "elapsed_time": int(elapsed_seconds),
        "description": description,
        "trainer": 1
    }
    # Only add distance if it's greater than 0
    if distance_meters > 0:
        payload["distance"] = distance_meters

    response = requests.post(
        "https://www.strava.com/api/v3/activities",
        headers={"Authorization": f"Bearer {access_token}"},
        data=payload
    )
    if response.status_code != 201:
        error_details = response.json() if response.status_code == 400 else response.text
        raise Exception(f"Strava rejected the upload with status {response.status_code}: {error_details}")
    return response.json()

def format_time(seconds):
    if seconds is None: return "--:--"
    seconds = int(seconds)
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    return f"{hours}:{minutes:02d}:{secs:02d}" if hours > 0 else f"{minutes}:{secs:02d}"

def prompt_for_value(field_name, current_value, value_type=float):
    display_val = current_value if current_value is not None else "not detected"
    new_val = input(f"  {field_name} [{display_val}]: ").strip()
    if new_val == '': return current_value
    try: return value_type(new_val)
    except ValueError: return current_value

# === Main Flow ===

print("📷 Select your spin bike screen photo:")
uploaded = files.upload()
if not uploaded: raise Exception("No file uploaded")

filename = list(uploaded.keys())[0]
image_bytes = uploaded[filename]

print(f"\n🔍 Extracting data from {filename}...")
workout = extract_workout_data(image_bytes)

print("\n" + "="*40 + "\n📊 EXTRACTED WORKOUT DATA\n" + "="*40)
print(f"  Time:      {format_time(workout.get('time_seconds'))}")
print(f"  Distance:  {workout.get('distance_miles')} mi")
print(f"  Calories:  {workout.get('calories')} kcal")
print(f"  Avg Watts: {workout.get('avg_watts')}")
print(f"  Avg RPM:   {workout.get('avg_rpm')}")
print(f"  Avg Speed: {workout.get('avg_speed_mph')} mph")
print("="*40)

workout['calories'] = prompt_for_value("Calories", workout.get('calories'), int)

edit = input("\n✏️  Edit any OTHER values? (y/n): ").strip().lower()
if edit == 'y':
    workout['distance_miles'] = prompt_for_value("Distance (mi)", workout.get('distance_miles'))
    workout['avg_watts'] = prompt_for_value("Avg Watts", workout.get('avg_watts'), int)
    workout['avg_rpm'] = prompt_for_value("Avg RPM", workout.get('avg_rpm'), int)

print("\n📅 When did this workout START? (e.g., 10am, today 9:30am, yesterday 6pm)")
while True:
    datetime_input = input("   Enter date/time: ").strip()
    start_dt = parse_datetime(datetime_input)
    if start_dt: break
    print("   ❌ Couldn't parse that.")

start_iso = start_dt.strftime("%Y-%m-%dT%H:%M:%S")
activity_name = "Equinox Spin Class"
description = " | ".join([f"{k}: {v}" for k, v in workout.items() if v is not None])
distance_meters = (workout.get('distance_miles') or 0) * 1609.34

print("\n" + "="*40 + "\n📤 READY TO UPLOAD\n" + "="*40)
print(f"  Name: {activity_name}\n  Start: {start_dt.strftime('%b %d, %Y at %I:%M %p')}")
print("="*40)

confirm = input("\n✅ Upload to Strava? (y/n): ").strip().lower()
if confirm == 'y':
    print("\n⏳ Uploading...")
    access_token = get_strava_access_token()
    result = create_strava_activity(access_token, activity_name, start_iso, workout.get('time_seconds', 0), distance_meters, description)
    print(f"\n🎉 Success! https://www.strava.com/activities/{result['id']}")
else:
    print("\n❌ Cancelled.")

📷 Select your spin bike screen photo:


Saving 260301.jpg to 260301 (1).jpg

🔍 Extracting data from 260301 (1).jpg...

📊 EXTRACTED WORKOUT DATA
  Time:      40:10
  Distance:  12.5 mi
  Calories:  402 kcal
  Avg Watts: 142
  Avg RPM:   62
  Avg Speed: 18.7 mph
  Calories [402]: 398

✏️  Edit any OTHER values? (y/n): n

📅 When did this workout START? (e.g., 10am, today 9:30am, yesterday 6pm)
   Enter date/time: today 10:00 am

📤 READY TO UPLOAD
  Name: Equinox Spin Class
  Start: Mar 01, 2026 at 10:00 AM

✅ Upload to Strava? (y/n): y

⏳ Uploading...

🎉 Success! https://www.strava.com/activities/17568223556


# 🔧 Strava Token Refresh Procedure

Use this when you get authentication errors or need to reauthorize Strava access.

### Step 1: Generate Authorization URL

Run this code block to generate your authorization URL:
```python
client_id = '188892'  # Your Strava Client ID
auth_url = f'https://www.strava.com/oauth/authorize?client_id={client_id}&response_type=code&redirect_uri=http://localhost&approval_prompt=force&scope=activity:write'
print(auth_url)
```

### Step 2: Authorize on Strava

1. Copy the URL from Step 1 and paste it into your browser
2. Click **"Authorize"** on the Strava page
3. You'll be redirected to: `http://localhost/?state=&code=XXXXXXXX`
4. Copy the code value (everything after `code=`)

### Step 3: Exchange Code for Refresh Token

Run this code with your new authorization code:
```python
import requests

client_id = '188892'
client_secret = '23ed63ba30c42e270a4a042a294ed724a55e67fd'
code = 'PASTE_YOUR_CODE_HERE'  # From Step 2

response = requests.post(
    'https://www.strava.com/oauth/token',
    data={
        'client_id': client_id,
        'client_secret': client_secret,
        'code': code,
        'grant_type': 'authorization_code'
    }
)

token_data = response.json()
print(f"New Refresh Token: {token_data['refresh_token']}")
print("\n✅ Copy this refresh token to your config.json file")
```

### Step 4: Update config.json

1. Open: `/content/drive/MyDrive/spin_to_strava/config.json`
2. Update the `strava_refresh_token` value with the token from Step 3
3. Save the file

### Step 5: Restart and Test

1. Runtime → Restart runtime
2. Run all cells
3. Upload a test photo to verify it works

---

**⚠️ Important Notes:**
- Authorization codes are single-use only
- If Step 3 fails, go back to Step 1 and get a fresh code
- The refresh token enables automatic token renewal
- Keep your config.json file secure and private

In [ ]:
client_id = '188892'  # Your Strava Client ID
auth_url = f'https://www.strava.com/oauth/authorize?client_id={client_id}&response_type=code&redirect_uri=http://localhost&approval_prompt=force&scope=activity:write'
print(auth_url)

https://www.strava.com/oauth/authorize?client_id=188892&response_type=code&redirect_uri=http://localhost&approval_prompt=force&scope=activity:write


79b88f5c0a89c3bedac1d5a8d74f316f2c8f185d

In [ ]:
import requests

client_id = '188892'
client_secret = '23ed63ba30c42e270a4a042a294ed724a55e67fd'
code = '79b88f5c0a89c3bedac1d5a8d74f316f2c8f185d'  # From Step 2

response = requests.post(
    'https://www.strava.com/oauth/token',
    data={
        'client_id': client_id,
        'client_secret': client_secret,
        'code': code,
        'grant_type': 'authorization_code'
    }
)

token_data = response.json()
print(f"New Refresh Token: {token_data['refresh_token']}")
print("\n✅ Copy this refresh token to your config.json file")

New Refresh Token: 1deb53f2e6e75c4f3b633a3eec6749dc17ba979b

✅ Copy this refresh token to your config.json file
